# BH Bayesian Optimization (Course Notebook)

This notebook is a beginner-friendly, step-by-step version of `BH.py`.

## Learning goals
- Understand pool-based Bayesian optimization (BO) on a real reaction dataset.
- Compare uncertainty estimates from Gaussian Process Regression (GPR) and Random Forest (RF).
- Run a batched Expected Improvement (EI) loop and interpret progress plots.

## How to use this notebook
1. Run cells from top to bottom once.
2. Change only the parameters in the **Edit here** cell first.
3. Re-run from the parameter cell downward after making changes.

## 1) Imports and display setup

This cell imports everything used later and configures plotting warnings so outputs stay clean.

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm, pearsonr, spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 2) Edit here: experiment settings

These are the only values students should edit on first pass.

In [ ]:
CSV_PATH = Path("BH_HTE_small.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path.cwd() / "BH_HTE_small.csv"

TARGET_COL = "Yield"
N_BO_STEPS = 10
BATCH_SIZE = 10
RANDOM_STATE = 7
N_INITIAL_LOW_POINTS = 10
MIN_NONZERO_LOW_POINTS = 5

np.random.seed(RANDOM_STATE)
print(f"CSV path: {CSV_PATH}")
print(f"BO steps: {N_BO_STEPS}, batch size: {BATCH_SIZE}, random seed: {RANDOM_STATE}")

## 3) Utility helpers

These helper functions keep later cells short and easier to read.

In [ ]:
def print_banner(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def show_current_figure() -> None:
    plt.tight_layout()
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            category=UserWarning,
            message="FigureCanvasAgg is non-interactive, and thus cannot be shown",
        )
        plt.show()
    plt.close()


def quick_inspect_csv(df: pd.DataFrame) -> None:
    print_banner("STEP 1A - QUICK CSV INSPECTION")
    n_rows, n_cols = df.shape
    print(f"Rows: {n_rows} | Columns: {n_cols}")
    print("\nFirst 5 rows:")
    print(df.head(5))
    print("\nMissing values per column:")
    print(df.isna().sum())
    if TARGET_COL in df.columns:
        print(f"\n{TARGET_COL} summary:")
        print(df[TARGET_COL].describe())

## 4) Model and BO helper functions

This cell defines preprocessing pipelines, uncertainty estimates, EI acquisition, and BO plotting.

In [ ]:
def gaussian_process_pipeline(cat_cols: list[str], num_cols: list[str]) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
            ("num", StandardScaler(), num_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )
    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * Matern(length_scale=1.0, length_scale_bounds=(0.2, 5.0), nu=1.5)
        + WhiteKernel(noise_level=0.2, noise_level_bounds=(0.01, 2.0))
    )
    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        alpha=1e-3,
        random_state=RANDOM_STATE,
        n_restarts_optimizer=6,
    )
    return Pipeline([("prep", preprocessor), ("model", model)])


def random_forest_pipeline(cat_cols: list[str], num_cols: list[str]) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
            ("num", StandardScaler(), num_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )
    model = RandomForestRegressor(
        n_estimators=50,
        random_state=RANDOM_STATE,
        min_samples_leaf=1,
        n_jobs=-1,
    )
    return Pipeline([("prep", preprocessor), ("model", model)])


def build_initial_low_yield_seed(df: pd.DataFrame, feature_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    if N_INITIAL_LOW_POINTS <= 0:
        raise ValueError("N_INITIAL_LOW_POINTS must be > 0.")

    df_sorted = df.sort_values(TARGET_COL, ascending=True).reset_index(drop=True)
    nonzero_low = df_sorted.loc[df_sorted[TARGET_COL] > 0].head(MIN_NONZERO_LOW_POINTS)
    if len(nonzero_low) < MIN_NONZERO_LOW_POINTS:
        raise ValueError(f"Need at least {MIN_NONZERO_LOW_POINTS} non-zero rows, found {len(nonzero_low)}.")
    if N_INITIAL_LOW_POINTS < MIN_NONZERO_LOW_POINTS:
        raise ValueError("N_INITIAL_LOW_POINTS must be >= MIN_NONZERO_LOW_POINTS.")

    seed_df = nonzero_low.copy()
    remaining_n = N_INITIAL_LOW_POINTS - len(seed_df)
    if remaining_n > 0:
        remaining = df_sorted.drop(index=seed_df.index, errors="ignore")
        seed_df = pd.concat([seed_df, remaining.head(remaining_n)], ignore_index=True)

    seed_df = seed_df.drop_duplicates(subset=feature_cols, keep="first").reset_index(drop=True)
    if len(seed_df) < N_INITIAL_LOW_POINTS:
        already = set(map(tuple, seed_df[feature_cols].to_numpy()))
        candidates = df_sorted.loc[
            ~df_sorted[feature_cols].apply(tuple, axis=1).isin(already)
        ].head(N_INITIAL_LOW_POINTS - len(seed_df))
        seed_df = pd.concat([seed_df, candidates], ignore_index=True)

    seed_df = seed_df.head(N_INITIAL_LOW_POINTS).reset_index(drop=True)
    seed_keys = set(map(tuple, seed_df[feature_cols].to_numpy()))
    candidate_df = df.loc[~df[feature_cols].apply(tuple, axis=1).isin(seed_keys), feature_cols].copy()
    return seed_df, candidate_df.reset_index(drop=True)


def lookup_true_yield(all_data: pd.DataFrame, feature_cols: list[str], chosen_features: pd.DataFrame) -> float:
    chosen_key = tuple(chosen_features.iloc[0][feature_cols].to_list())
    all_keys = all_data[feature_cols].apply(tuple, axis=1)
    matches = all_data.loc[all_keys == chosen_key, TARGET_COL]
    if matches.empty:
        raise ValueError(f"Chosen condition not found in dataset: {chosen_key}")
    return float(matches.iloc[0])


def rf_bootstrap_predict_mean_std(
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_pool: pd.DataFrame,
    cat_cols: list[str],
    num_cols: list[str],
) -> Tuple[np.ndarray, np.ndarray]:
    n_rows = len(x_train)
    rng = np.random.RandomState(RANDOM_STATE)
    all_tree_predictions = []

    for _ in range(2):
        sampled_idx = rng.choice(n_rows, size=n_rows, replace=True)
        x_boot = x_train.iloc[sampled_idx].reset_index(drop=True)
        y_boot = y_train.iloc[sampled_idx].reset_index(drop=True)
        rf_pipe = random_forest_pipeline(cat_cols, num_cols)
        rf_pipe.fit(x_boot, y_boot)
        x_pool_trans = rf_pipe.named_steps["prep"].transform(x_pool)
        for tree in rf_pipe.named_steps["model"].estimators_:
            all_tree_predictions.append(tree.predict(x_pool_trans))

    all_tree_predictions = np.asarray(all_tree_predictions)
    return all_tree_predictions.mean(axis=0), all_tree_predictions.std(axis=0, ddof=1)


def fit_predict_with_uncertainty(
    model_kind: str,
    model_pipeline: Pipeline,
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_pool: pd.DataFrame,
    print_diagnostics: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    if model_kind == "gpr":
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=ConvergenceWarning)
            model_pipeline.fit(x_train, y_train)
        mu, sd = model_pipeline.predict(x_pool, return_std=True)
        if print_diagnostics:
            gp_model = model_pipeline.named_steps["model"]
            print(
                "[GPR diagnostics] "
                f"kernel={gp_model.kernel_} | "
                f"mu_std={np.std(mu):.3f} | "
                f"mu_range=({np.min(mu):.3f}, {np.max(mu):.3f}) | "
                f"sd_range=({np.min(sd):.3f}, {np.max(sd):.3f})"
            )
        return mu.astype(float), sd.astype(float)
    if model_kind == "rf":
        preprocessor = model_pipeline.named_steps["prep"]
        transformers = preprocessor.transformers
        cat_cols_local = list(transformers[0][2])
        num_cols_local = list(transformers[1][2])
        mu, sd = rf_bootstrap_predict_mean_std(x_train, y_train, x_pool, cat_cols_local, num_cols_local)
        return mu.astype(float), sd.astype(float)
    raise ValueError("Unsupported model_kind. Use 'gpr' or 'rf'.")


def expected_improvement(mu: np.ndarray, sigma: np.ndarray, best_so_far: float, xi: float = 0.01) -> np.ndarray:
    sigma_safe = np.maximum(sigma, 1e-12)
    improvement = mu - best_so_far - xi
    z = improvement / sigma_safe
    ei = improvement * norm.cdf(z) + sigma_safe * norm.pdf(z)
    ei[sigma <= 1e-12] = 0.0
    return ei


def plot_iteration_diagnostics(
    ranking: pd.DataFrame,
    selected_batch: pd.DataFrame,
    feature_cols: list[str],
    step: int,
    best_so_far: float,
    batch_size: int,
) -> None:
    selected_keys = set(selected_batch[feature_cols].apply(tuple, axis=1).tolist())
    ranking_keys = ranking[feature_cols].apply(tuple, axis=1)
    selected_mask = ranking_keys.isin(selected_keys)

    plt.figure(figsize=(8, 6))
    sc = plt.scatter(
        ranking["pred_std"],
        ranking["pred_mean"],
        c=ranking["EI"],
        cmap="viridis",
        alpha=0.75,
        s=40,
    )
    plt.scatter(
        ranking.loc[selected_mask, "pred_std"],
        ranking.loc[selected_mask, "pred_mean"],
        facecolors="none",
        edgecolors="red",
        linewidths=1.5,
        s=90,
        zorder=5,
        label=f"Selected batch (top {batch_size} EI)",
    )
    plt.colorbar(sc, label="Expected Improvement (EI)")
    plt.axhline(best_so_far, linestyle="--", linewidth=1, color="gray", label="Current best observed")
    plt.xlabel("Predicted uncertainty (std)")
    plt.ylabel("Predicted mean yield")
    plt.title(f"Iteration {step}: uncertainty vs predicted mean (batch-highlighted)")
    plt.legend()
    show_current_figure()

    top_k = ranking.head(max(20, batch_size)).copy()
    top_k_keys = top_k[feature_cols].apply(tuple, axis=1)
    top_k_selected = top_k_keys.isin(selected_keys)
    label_cols = feature_cols[:4]
    labels = [" | ".join([f"{col}={row[col]}" for col in label_cols]) for _, row in top_k.iterrows()]
    bar_colors = ["tomato" if is_sel else "steelblue" for is_sel in top_k_selected]
    plt.figure(figsize=(11, 5))
    plt.bar(range(len(top_k)), top_k["EI"], color=bar_colors)
    plt.xticks(range(len(top_k)), labels, rotation=45, ha="right")
    plt.ylabel("Expected Improvement (EI)")
    plt.xlabel("Top candidates (highest EI on left)")
    plt.title(f"Iteration {step}: top candidates by EI (selected batch in red)")
    show_current_figure()

## 5) Load and inspect the dataset

Check that data loads correctly, inspect missing values, and separate feature columns from target.

In [ ]:
print_banner("STEP 1 - LOAD DATA")
df = pd.read_csv(CSV_PATH)
print(f"Loaded CSV from: {CSV_PATH}")
quick_inspect_csv(df)

print_banner("STEP 1B - FULL DATA INSPECTION")
print(f"Shape: {df.shape}")
print("\nData types:")
print(df.dtypes)
print("\nFirst 10 rows:")
print(df.head(10))
print("\nMissing values per column:")
print(df.isna().sum())

feature_cols = [col for col in df.columns if col != TARGET_COL]
cat_cols = [col for col in feature_cols if df[col].dtype == "object"]
num_cols = [col for col in feature_cols if col not in cat_cols]
print(f"\nTarget column: {TARGET_COL}")
print(f"Feature columns: {feature_cols}")
print(f"Categorical columns: {cat_cols}")
print(f"Numeric columns: {num_cols}")

## 6) Build BO universe, low-yield seed, and remaining pool

We intentionally start from poor outcomes so students can see BO recover toward better conditions.

In [ ]:
print_banner("STEP 3 - DATASET-SCOPED POOL (NO SYNTHETIC EXPANSION)")
universe_df = df[feature_cols + [TARGET_COL]].drop_duplicates(subset=feature_cols, keep="first").reset_index(drop=True)
print("Universe built from dataset rows (first 10):")
print(universe_df.head(10))
print(f"Unique feature combinations in universe: {len(universe_df)}")

print_banner("STEP 4 - LOW-YIELD INITIALIZATION + REMAINING CANDIDATES")
observed_df, candidate_df = build_initial_low_yield_seed(universe_df, feature_cols)
n_nonzero_seed = int((observed_df[TARGET_COL] > 0).sum())
print(f"Initial low-yield seed size: {len(observed_df)} (non-zero points in seed: {n_nonzero_seed})")
print(f"Constraint check: at least {MIN_NONZERO_LOW_POINTS} non-zero low-yield points => {n_nonzero_seed >= MIN_NONZERO_LOW_POINTS}")
print(f"Remaining candidates available for BO: {len(candidate_df)}")
print("\nObserved training rows (head):")
print(observed_df.head())
print("\nCandidate rows (head):")
print(candidate_df.head())

## 7) Diagnostic check: GPR vs RF before optimization

This section compares predicted means and uncertainties from both models on the same pool.

In [ ]:
print_banner("STEP 5 - MODEL DIAGNOSTIC: GPR VS RF UNCERTAINTY")
x_train = observed_df[feature_cols]
y_train = observed_df[TARGET_COL]

gpr_pipe = gaussian_process_pipeline(cat_cols, num_cols)
rf_pipe = random_forest_pipeline(cat_cols, num_cols)
mu_gpr, sd_gpr = fit_predict_with_uncertainty("gpr", gpr_pipe, x_train, y_train, candidate_df)
mu_rf, sd_rf = fit_predict_with_uncertainty("rf", rf_pipe, x_train, y_train, candidate_df)

diagnostic = candidate_df.copy()
diagnostic["mu_gpr"] = mu_gpr
diagnostic["sd_gpr"] = sd_gpr
diagnostic["mu_rf"] = mu_rf
diagnostic["sd_rf"] = sd_rf
diagnostic["abs_mu_diff"] = np.abs(diagnostic["mu_gpr"] - diagnostic["mu_rf"])
diagnostic["abs_sd_diff"] = np.abs(diagnostic["sd_gpr"] - diagnostic["sd_rf"])

print("Top 10 points where model means disagree the most:")
print(diagnostic.sort_values("abs_mu_diff", ascending=False).head(10))
print("\nTop 10 points where uncertainties disagree the most:")
print(diagnostic.sort_values("abs_sd_diff", ascending=False).head(10))

plt.figure(figsize=(8, 6))
plt.scatter(diagnostic["mu_gpr"], diagnostic["mu_rf"], alpha=0.5)
low = min(diagnostic["mu_gpr"].min(), diagnostic["mu_rf"].min())
high = max(diagnostic["mu_gpr"].max(), diagnostic["mu_rf"].max())
plt.plot([low, high], [low, high], "--", linewidth=1)
plt.xlabel("GPR predicted mean yield")
plt.ylabel("RF predicted mean yield")
plt.title("Model diagnostic: predicted mean comparison")
show_current_figure()

plt.figure(figsize=(8, 6))
plt.scatter(diagnostic["sd_gpr"], diagnostic["sd_rf"], alpha=0.5)
low = min(diagnostic["sd_gpr"].min(), diagnostic["sd_rf"].min())
high = max(diagnostic["sd_gpr"].max(), diagnostic["sd_rf"].max())
plt.plot([low, high], [low, high], "--", linewidth=1)
plt.xlabel("GPR uncertainty (std)")
plt.ylabel("RF uncertainty proxy (std)")
plt.title("Model diagnostic: uncertainty comparison")
show_current_figure()

## 8) 5-fold CV comparison (GPR vs RF)

This gives students a basic view of prediction quality and whether uncertainty tracks error.

In [ ]:
print_banner("STEP 5B - 5-FOLD CV: TEST PREDICTION COMPARISON (GPR VS RF)")
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y_true_all, y_pred_gpr_all, y_pred_rf_all = [], [], []
y_std_gpr_all, y_std_rf_all = [], []
fold_rows = []

for fold_idx, (train_idx, test_idx) in enumerate(kf.split(observed_df), start=1):
    train_fold = observed_df.iloc[train_idx].reset_index(drop=True)
    test_fold = observed_df.iloc[test_idx].reset_index(drop=True)
    x_train_fold = train_fold[feature_cols]
    y_train_fold = train_fold[TARGET_COL]
    x_test_fold = test_fold[feature_cols]
    y_test_fold = test_fold[TARGET_COL].to_numpy(dtype=float)

    gpr_fold = gaussian_process_pipeline(cat_cols, num_cols)
    rf_fold = random_forest_pipeline(cat_cols, num_cols)
    pred_gpr_fold, std_gpr_fold = fit_predict_with_uncertainty("gpr", gpr_fold, x_train_fold, y_train_fold, x_test_fold, print_diagnostics=False)
    pred_rf_fold, std_rf_fold = fit_predict_with_uncertainty("rf", rf_fold, x_train_fold, y_train_fold, x_test_fold, print_diagnostics=False)

    y_true_all.extend(y_test_fold.tolist())
    y_pred_gpr_all.extend(pred_gpr_fold.tolist())
    y_pred_rf_all.extend(pred_rf_fold.tolist())
    y_std_gpr_all.extend(std_gpr_fold.tolist())
    y_std_rf_all.extend(std_rf_fold.tolist())

    fold_rows.append(
        {
            "fold": fold_idx,
            "n_test": len(test_fold),
            "mae_gpr": mean_absolute_error(y_test_fold, pred_gpr_fold),
            "mae_rf": mean_absolute_error(y_test_fold, pred_rf_fold),
            "r2_gpr": r2_score(y_test_fold, pred_gpr_fold),
            "r2_rf": r2_score(y_test_fold, pred_rf_fold),
        }
    )

cv_df = pd.DataFrame(fold_rows)
print("Per-fold test metrics:")
print(cv_df)
print("\nMean CV metrics:")
print(cv_df[["mae_gpr", "mae_rf", "r2_gpr", "r2_rf"]].mean().to_frame("mean"))

cv_pred_df = pd.DataFrame(
    {
        "y_true": np.asarray(y_true_all, dtype=float),
        "y_pred_gpr": np.asarray(y_pred_gpr_all, dtype=float),
        "y_pred_rf": np.asarray(y_pred_rf_all, dtype=float),
        "std_gpr": np.asarray(y_std_gpr_all, dtype=float),
        "std_rf": np.asarray(y_std_rf_all, dtype=float),
    }
)
cv_pred_df["abs_err_gpr"] = np.abs(cv_pred_df["y_true"] - cv_pred_df["y_pred_gpr"])
cv_pred_df["abs_err_rf"] = np.abs(cv_pred_df["y_true"] - cv_pred_df["y_pred_rf"])

gpr_pearson_r, gpr_pearson_p = pearsonr(cv_pred_df["std_gpr"], cv_pred_df["abs_err_gpr"])
gpr_spearman_rho, gpr_spearman_p = spearmanr(cv_pred_df["std_gpr"], cv_pred_df["abs_err_gpr"])
rf_pearson_r, rf_pearson_p = pearsonr(cv_pred_df["std_rf"], cv_pred_df["abs_err_rf"])
rf_spearman_rho, rf_spearman_p = spearmanr(cv_pred_df["std_rf"], cv_pred_df["abs_err_rf"])
print("\nUncertainty vs error correlation on test folds:")
print(f"GPR: Pearson r={gpr_pearson_r:.3f} (p={gpr_pearson_p:.2e}), Spearman rho={gpr_spearman_rho:.3f} (p={gpr_spearman_p:.2e})")
print(f"RF : Pearson r={rf_pearson_r:.3f} (p={rf_pearson_p:.2e}), Spearman rho={rf_spearman_rho:.3f} (p={rf_spearman_p:.2e})")

plt.figure(figsize=(8, 6))
plt.scatter(cv_pred_df["y_true"], cv_pred_df["y_pred_gpr"], alpha=0.7, label="GPR")
plt.scatter(cv_pred_df["y_true"], cv_pred_df["y_pred_rf"], alpha=0.7, label="RF")
low = min(cv_pred_df["y_true"].min(), cv_pred_df["y_pred_gpr"].min(), cv_pred_df["y_pred_rf"].min())
high = max(cv_pred_df["y_true"].max(), cv_pred_df["y_pred_gpr"].max(), cv_pred_df["y_pred_rf"].max())
plt.plot([low, high], [low, high], "--", linewidth=1, color="black")
plt.xlabel("True yield (test folds)")
plt.ylabel("Predicted yield")
plt.title("5-fold CV test predictions: GPR vs RF")
plt.legend()
show_current_figure()

## 9) Batched BO loop (EI with GPR)

At each iteration: fit GPR on observed data, rank pool by EI, pick top batch, reveal true yields, and update train/pool.

In [ ]:
print_banner("STEP 6 - BATCHED SEQUENTIAL BO")
bo_train = observed_df.copy()
bo_pool = candidate_df.copy()
selected_history = []

def build_ranking_table(current_train: pd.DataFrame, current_pool: pd.DataFrame) -> tuple[pd.DataFrame, float]:
    x_train_local = current_train[feature_cols]
    y_train_local = current_train[TARGET_COL]
    gpr_local = gaussian_process_pipeline(cat_cols, num_cols)
    mu_local, sd_local = fit_predict_with_uncertainty("gpr", gpr_local, x_train_local, y_train_local, current_pool)
    best_local = float(y_train_local.max())
    ei_local = expected_improvement(mu_local, sd_local, best_local, xi=0.01)
    ranking_local = current_pool.copy()
    ranking_local["pred_mean"] = mu_local
    ranking_local["pred_std"] = sd_local
    ranking_local["EI"] = ei_local
    ranking_local = ranking_local.sort_values("EI", ascending=False).reset_index(drop=True)
    return ranking_local, best_local

def choose_batch_and_record(ranking_local: pd.DataFrame, step_local: int, batch_size_local: int):
    batch = ranking_local.head(batch_size_local).copy().reset_index(drop=True)
    chosen_records_local, chosen_keys_local, history_items_local = [], [], []
    for batch_rank, (_, chosen_local) in enumerate(batch.iterrows(), start=1):
        chosen_features_local = chosen_local[feature_cols].to_frame().T
        chosen_key_local = tuple(chosen_features_local.iloc[0][feature_cols].to_list())
        observed_yield_local = lookup_true_yield(universe_df, feature_cols, chosen_features_local)
        chosen_record_local = chosen_features_local.copy()
        chosen_record_local[TARGET_COL] = observed_yield_local
        chosen_record_local["source"] = f"bo_step_{step_local}_batch_{batch_rank}_dataset_revealed"
        chosen_records_local.append(chosen_record_local)
        chosen_keys_local.append(chosen_key_local)
        history_items_local.append(
            {
                "step": step_local,
                "batch_rank": batch_rank,
                "best_before": np.nan,
                "pred_mean": float(chosen_local["pred_mean"]),
                "pred_std": float(chosen_local["pred_std"]),
                "EI": float(chosen_local["EI"]),
                TARGET_COL: observed_yield_local,
                **{col: chosen_local[col] for col in feature_cols},
            }
        )
    chosen_records_df = pd.concat(chosen_records_local, ignore_index=True)
    return chosen_records_df, chosen_keys_local, history_items_local, batch

def update_train_and_pool(current_train, current_pool, chosen_record_local, chosen_keys_local):
    new_train = pd.concat([current_train, chosen_record_local], ignore_index=True)
    pool_keys_local = current_pool[feature_cols].apply(tuple, axis=1)
    new_pool = current_pool.loc[~pool_keys_local.isin(set(chosen_keys_local))].reset_index(drop=True)
    return new_train, new_pool

for step in range(1, N_BO_STEPS + 1):
    if len(bo_pool) == 0:
        print("Pool is empty; stopping early.")
        break
    ranking, best_so_far = build_ranking_table(bo_train, bo_pool)
    current_batch_size = min(BATCH_SIZE, len(ranking))
    print_banner(f"BO ITERATION {step}")
    print(f"Current best observed yield: {best_so_far:.3f}")
    print(f"Batch size this iteration: {current_batch_size}")
    print("Top 5 candidates by EI:")
    print(ranking.head(5))

    chosen_records, chosen_keys, history_items, selected_batch = choose_batch_and_record(ranking, step, current_batch_size)
    for history_item in history_items:
        history_item["best_before"] = best_so_far

    plot_iteration_diagnostics(ranking, selected_batch, feature_cols, step, best_so_far, current_batch_size)

    batch_yields = [float(item[TARGET_COL]) for item in history_items]
    print(f"Batch predicted means range: {float(ranking.head(current_batch_size)['pred_mean'].min()):.3f} to {float(ranking.head(current_batch_size)['pred_mean'].max()):.3f}")
    print(f"Revealed true yields range: {min(batch_yields):.3f} to {max(batch_yields):.3f}")

    selected_history.extend(history_items)
    bo_train, bo_pool = update_train_and_pool(bo_train, bo_pool, chosen_records, chosen_keys)
    print(f"Training set size is now: {len(bo_train)}")
    print(f"Pool size is now: {len(bo_pool)}")

## 10) Final BO history and trajectory plots

This section summarizes what BO learned over time.

In [ ]:
history_df = pd.DataFrame(selected_history)
print_banner("STEP 7 - BO HISTORY TABLES")
print("Full BO history:")
print(history_df)
print("\nYield progression (best_before vs chosen observed yield):")
print(history_df[["step", "best_before", TARGET_COL, "pred_std", "EI"]])

step_summary = (
    history_df.groupby("step", as_index=False)
    .agg(
        best_before=("best_before", "first"),
        observed_mean=(TARGET_COL, "mean"),
        observed_std=(TARGET_COL, "std"),
        ei_mean=("EI", "mean"),
        ei_std=("EI", "std"),
        pred_std_mean=("pred_std", "mean"),
        pred_std_std=("pred_std", "std"),
        n_points=("EI", "size"),
    )
    .sort_values("step")
    .reset_index(drop=True)
)
step_summary["observed_std"] = step_summary["observed_std"].fillna(0.0)
step_summary["ei_std"] = step_summary["ei_std"].fillna(0.0)
step_summary["pred_std_std"] = step_summary["pred_std_std"].fillna(0.0)

plt.figure(figsize=(8, 5))
plt.plot(step_summary["step"], step_summary["best_before"], marker="o", label="Best observed before step")
plt.errorbar(step_summary["step"], step_summary["observed_mean"], yerr=step_summary["observed_std"], fmt="s-", capsize=4, label="Observed yield (mean +/- std within batch)")
plt.xlabel("BO step")
plt.ylabel("Yield")
plt.title("Sequential BO trajectory (batch-aware)")
plt.legend()
show_current_figure()

plt.figure(figsize=(8, 5))
plt.errorbar(step_summary["step"], step_summary["ei_mean"], yerr=step_summary["ei_std"], fmt="o-", capsize=4, label="EI (mean +/- std within batch)")
plt.xlabel("BO step")
plt.ylabel("Expected Improvement")
plt.title("Acquisition value per BO step")
plt.legend()
show_current_figure()

plt.figure(figsize=(8, 5))
plt.errorbar(step_summary["step"], step_summary["pred_std_mean"], yerr=step_summary["pred_std_std"], fmt="o-", capsize=4, label="Predicted std (mean +/- std within batch)")
plt.xlabel("BO step")
plt.ylabel("Predicted uncertainty (std)")
plt.title("Uncertainty per BO step")
plt.legend()
show_current_figure()

## 11) Try it yourself

1. Increase `N_BO_STEPS` and check whether best yield plateaus.
2. Decrease `BATCH_SIZE` (e.g., 3) and compare exploration behavior.
3. Change `N_INITIAL_LOW_POINTS` and observe how initialization quality changes trajectory.

After each change, re-run from the **Edit here** cell to the end.